# AnimalCLEF 2026: Global + Local Features Ensemble

**Approach**: Combines global features (MiewID v3) with species-specific local features (SIFT, SuperPoint, ALIKED, DISK) using weighted ensemble voting.

**Expected Improvements**:
- SeaTurtleID2022: 38% → 50-60% known match rate
- Better clustering for deformable bodies (Salamanders) and dense patterns (Texas Lizards)

**Runtime**: ~90 minutes (within Kaggle's 9-hour limit)

## Section 1: Setup and Installation

In [ ]:
# Cell 1.1: Install Dependencies
!pip install kornia kornia-rs kornia-moons --quiet
!pip install wildlife-datasets wildlife-tools timm scikit-learn --quiet --upgrade
# Use transformers 4.36.0 for MiewID v3 compatibility
!pip install transformers==4.36.0 --quiet

import kornia
print(f"✓ Kornia {kornia.__version__} installed")

import transformers
print(f"✓ Transformers {transformers.__version__} installed")
print("✓ All dependencies installed successfully")

In [ ]:
# Cell 1.2: Imports
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import normalize
import cv2
from PIL import Image

# Deep learning
import timm
from transformers import AutoModel

# Local features (kornia for all extractors)
import kornia.feature as KF

# Dataset
from wildlife_datasets.datasets import AnimalCLEF2026

warnings.filterwarnings("ignore")
print("✓ Imports successful")

In [ ]:
# Cell 1.3: Configuration - Species-Specific Weights

# V2.8: yellow-spot + specular mask for SalamanderID2025; CALIB_KPT_CAP=300
# SalamanderID2025 and LynxID2025 use SIFT-only (deformable / IR camera trap)

SPECIES_CONFIG = {
    "SalamanderID2025": {
        # Deformable bodies → SIFT + KAZE (SAM3 100% coverage → masked keypoints)
        "global_weight": 0.65,
        "local_extractors": ["sift", "kaze"],
        "local_weights": {"sift": 0.20, "kaze": 0.15},
        "threshold_known": 0.40,
        "threshold_cluster": 0.35,
        "image_size": 512,
        "qe_k": 3,
    },
    "SeaTurtleID2022": {
        # Rigid, high-contrast features → SIFT + KAZE (non-linear scale space)
        "global_weight": 0.65,
        "local_extractors": ["sift", "kaze"],
        "local_weights": {"sift": 0.20, "kaze": 0.15},
        "threshold_known": 0.45,
        "threshold_cluster": 0.40,
        "image_size": 512,
        "qe_k": 8,
    },
    "LynxID2025": {
        # Rosette/fur patterns → SIFT + KAZE (0% SAM3 → on originals; conservative weight)
        "global_weight": 0.70,
        "local_extractors": ["sift", "kaze"],
        "local_weights": {"sift": 0.20, "kaze": 0.10},
        "threshold_known": 0.40,
        "threshold_cluster": 0.35,
        "image_size": 512,
        "qe_k": 5,
    },
    "TexasHornedLizards": {
        # Dense spot patterns → SIFT + KAZE (non-linear scale space)
        "global_weight": 0.65,
        "local_extractors": ["sift", "kaze"],
        "local_weights": {"sift": 0.20, "kaze": 0.15},
        "threshold_known": None,  # Zero-shot
        "threshold_cluster": 0.30,
        "image_size": 512,
        "qe_k": 5,
    },
}

# Verify weights sum to 1.0
for species, cfg in SPECIES_CONFIG.items():
    total = cfg["global_weight"] + sum(cfg["local_weights"].values())
    assert abs(total - 1.0) < 0.01, f"{species} weights don't sum to 1.0: {total}"

print("✓ Species configuration loaded")
for species, cfg in SPECIES_CONFIG.items():
    extractors_str = " + ".join(
        f"{k.upper()}={v:.0%}" for k, v in cfg["local_weights"].items()
    )
    print(f"  {species}: Global {cfg['global_weight']:.0%}, {extractors_str}")

In [ ]:
# Cell 1.4: Device Detection

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
NUM_WORKERS = 4
ROOT_DIR = "/kaggle/input/animal-clef-2026"

print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Count: {torch.cuda.device_count()}")
    if torch.cuda.device_count() > 1:
        print(f"🚀 Using {torch.cuda.device_count()} GPUs") 

# Create cache directory
os.makedirs("cache/embeddings", exist_ok=True)
os.makedirs("cache/local_features", exist_ok=True)
os.makedirs("cache/match_scores", exist_ok=True)
print("✓ Cache directories created")

In [ ]:
# V2.5 Cell 1.4b: Preload KAZE cache from sreevaatsavbavana/v2-5-cache dataset
# ─────────────────────────────────────────────────────────────────────────────
# The v2-5-cache dataset contains precomputed KAZE local features and match
# scores for SalamanderID2025 and LynxID2025.  Copying them here means the
# existing cache-check logic in Cells 3.5 and 4.4 will load them directly
# without re-extracting.

import shutil

# Kaggle mounts datasets at /kaggle/input/{slug}/; try both naming conventions
_V25_CANDIDATES = [
    "/kaggle/input/datasets/sreevaatsavbavana/v2-5-cache/cache",
    "/kaggle/input/v2-5-cache/cache",
    "/kaggle/input/v2-5-cache/cache",
]

_V25_SRC = None
for _p in _V25_CANDIDATES:
    if os.path.exists(_p):
        _V25_SRC = _p
        break

if _V25_SRC:
    print(f"V2.5 cache found: {_V25_SRC}")
    _copied = 0
    for _sub in ["local_features", "match_scores"]:
        _src_dir = os.path.join(_V25_SRC, _sub)
        _dst_dir = os.path.join("cache", _sub)
        if os.path.isdir(_src_dir):
            for _fname in sorted(os.listdir(_src_dir)):
                # V2.8: Salamander features recomputed with yellow+specular mask
                if _fname.startswith('SalamanderID2025'):
                    print(f"  Skip    {_sub}/{_fname}  (V2.8 mask → recompute)")
                    continue
                _src_f = os.path.join(_src_dir, _fname)
                _dst_f = os.path.join(_dst_dir, _fname)
                if not os.path.exists(_dst_f):
                    shutil.copy2(_src_f, _dst_f)
                    print(f"  Copied  {_sub}/{_fname}")
                    _copied += 1
                else:
                    print(f"  Present {_sub}/{_fname}")
    print(f"✓ V2.5 cache preloaded ({_copied} new files)")
else:
    print("⚠ V2.5 cache dataset not mounted — KAZE features will be extracted from scratch")


In [ ]:
# Cell 1.5: Load Data

print("📂 Loading AnimalCLEF 2026 dataset...")
full_dataset = AnimalCLEF2026(root=ROOT_DIR, transform=None, load_label=True)
metadata = full_dataset.metadata

test_meta = metadata[metadata["split"] == "test"]
train_meta = metadata[metadata["split"] == "train"]

print(f"✓ Dataset loaded")
print(f"  Total images: {len(metadata):,}")
print(f"  Test images: {len(test_meta):,}")
print(f"  Train images: {len(train_meta):,}")
print(f"\nSpecies breakdown:")
for species in test_meta["dataset"].unique():
    n_test = len(test_meta[test_meta["dataset"] == species])
    n_train = len(train_meta[train_meta["dataset"] == species])
    print(f"  {species}: {n_test} test, {n_train} train")

In [ ]:
# Cell 1.6: SAM 3 — build SEG_MAP and keypoint-mask helper
#
# Strategy (V1.3): run SIFT on the ORIGINAL image, then discard keypoints
# that land on background pixels.  This avoids the white-boundary artifacts
# that hurt V1.2 (score 0.26330 vs V1 baseline 0.30655).
#
# LynxID2025 has 0% cache coverage → get_seg_mask() returns None → no change
# for that species.  All other species get filtered keypoints.
from pathlib import Path
import cv2
import numpy as np

SEG_CACHE_ROOT = Path('/kaggle/input/datasets/sreevaatsavbavana/animalclef-26-sam3/sam3_yolo_output/segmented_images')
_root = Path(ROOT_DIR)  # ROOT_DIR defined in Cell 1.4 as a str

# Key: relative path  e.g. 'SeaTurtleID2022/img001.jpg'
SEG_MAP = {}  # {rel_key: Path_to_segmented_jpg}

all_meta = pd.concat([train_meta, test_meta])
for _, row in tqdm(all_meta.iterrows(), total=len(all_meta), desc='Building SEG_MAP'):
    stem    = Path(row['path']).stem
    ds_name = row['dataset']
    rel_key = str(row['path'])
    cached  = SEG_CACHE_ROOT / ds_name / (stem + '.jpg')
    if cached.exists():
        SEG_MAP[rel_key] = cached

n_total    = len(all_meta)
n_cached   = len(SEG_MAP)
n_fallback = n_total - n_cached
print(f'SEG_MAP built: {n_cached:,} cached  |  {n_fallback:,} fallback to original  |  {n_total:,} total')

# Per-species / per-split breakdown
print()
print(f'{"Dataset":<25} {"split":<6} {"cached":>7} {"total":>7} {"coverage":>9}')
print('-' * 58)
for ds in sorted(all_meta["dataset"].unique()):
    for split, meta_split in [("train", train_meta), ("test", test_meta)]:
        rows = meta_split[meta_split["dataset"] == ds]
        if len(rows) == 0:
            continue
        n_hit = sum(1 for _, r in rows.iterrows() if str(r["path"]) in SEG_MAP)
        pct   = 100 * n_hit / len(rows)
        flag  = "" if pct == 100 else " ⚠" if pct < 50 else " ✓"
        print(f'  {ds:<23} {split:<6} {n_hit:>7,} {len(rows):>7,} {pct:>8.1f}%{flag}')

def get_seg_mask(img_path):
    """
    Return a binary uint8 mask (1=animal, 0=background) derived from the
    pre-segmented image, or None if not in the cache.

    The segmented image has background pixels set to pure white (255,255,255).
    Mask = pixels that are NOT pure white.  Imperfect for white-furred/scaled
    animals, but good enough for SeaTurtles, Salamanders, and TexasLizards.
    """
    p = Path(img_path)
    try:
        rel_key = str(p.relative_to(_root))
    except ValueError:
        return None
    if rel_key not in SEG_MAP:
        return None
    seg = cv2.imread(str(SEG_MAP[rel_key]))
    if seg is None:
        return None
    # Background was painted to (255, 255, 255) — mark those as 0
    is_bg = (seg[:, :, 0] == 255) & (seg[:, :, 1] == 255) & (seg[:, :, 2] == 255)
    mask  = (~is_bg).astype(np.uint8)
    # ── V2.8: Salamander yellow-spot + specular filter ────────────────────
    # Flash photography creates specular hotspots (~27% of SIFT kpts) that
    # carry no identity information.  Yellow spots are the actual per-
    # individual marker on fire salamanders.
    if 'SalamanderID2025' in rel_key:
        orig = cv2.imread(str(p))
        if orig is not None:
            hsv      = cv2.cvtColor(orig, cv2.COLOR_BGR2HSV)
            specular = (hsv[:, :, 2] > 220) & (hsv[:, :, 1] < 40)
            yellow   = ((hsv[:, :, 0] >= 18) & (hsv[:, :, 0] <= 42) &
                        (hsv[:, :, 1] >  80) & (hsv[:, :, 2] >  80))
            _k       = np.ones((25, 25), np.uint8)
            yellow_d = cv2.dilate(yellow.astype(np.uint8), _k).astype(bool)
            no_spec  = mask.astype(bool) & ~specular
            mask     = (no_spec & yellow_d).astype(np.uint8) if yellow_d.sum() >= 50 \
                       else no_spec.astype(np.uint8)
    # ─────────────────────────────────────────────────────────────────────
    return mask

print('get_seg_mask() ready.')


## Section 2: Global Features (MiewID v3)

In [ ]:
# Cell 2.1: MiewID v3 Model

class MiewIDWrapper(nn.Module):
    """MiewID v3 wrapper for global feature extraction."""
    def __init__(self):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(
            "conservationxlabs/miewid-msv3",
            trust_remote_code=True
        )

    def forward(self, x):
        return self.backbone(x)


def get_global_model():
    """Initialize MiewID v3 model with multi-GPU support."""
    model = MiewIDWrapper()
    model.to(DEVICE)
    
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    
    model.eval()
    return model

print("✓ MiewID v3 model wrapper defined")

In [ ]:
# Cell 2.2: Extract Global Features

def extract_global_features(model, dataset, image_size):
    """Extract L2-normalized global features with TTA (horizontal flip)."""
    dataset.transform = T.Compose([
        T.Resize((image_size, image_size)),
        T.ToTensor(),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])
    
    loader = DataLoader(
        dataset, 
        batch_size=BATCH_SIZE, 
        num_workers=NUM_WORKERS, 
        shuffle=False
    )
    
    all_features = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Extracting global features", leave=False):
            images = batch[0].to(DEVICE)
            
            # TTA: original + horizontal flip
            with torch.cuda.amp.autocast():
                feats_sum = model(images) + model(torch.flip(images, dims=[3]))
            
            # L2 normalize
            feats_norm = torch.nn.functional.normalize(feats_sum.float(), p=2, dim=1)
            all_features.append(feats_norm.cpu().numpy())
    
    return np.concatenate(all_features)

print("✓ Global feature extraction function defined")

In [ ]:
# Cell 2.3: Cache Global Embeddings

global_features_cache = {}
model = get_global_model()

for species in test_meta["dataset"].unique():
    print(f"\n{'='*60}")
    print(f"Processing {species} - Global Features")
    
    cfg = SPECIES_CONFIG[species]
    cache_file = f"cache/embeddings/{species}_global.npy"
    
    # Check cache
    if os.path.exists(cache_file):
        print(f"  Loading cached embeddings: {cache_file}")
        features = np.load(cache_file)
    else:
        # Extract features
        sp_meta = test_meta[test_meta["dataset"] == species]
        sp_dataset = full_dataset.get_subset(sp_meta.index.values)
        
        features = extract_global_features(model, sp_dataset, cfg["image_size"])
        
        # Cache to disk
        np.save(cache_file, features)
        print(f"  Cached to {cache_file}")
    
    global_features_cache[species] = features
    print(f"  Shape: {features.shape}, Norm: {np.linalg.norm(features[0]):.3f}")

del model
torch.cuda.empty_cache()
print("\n✓ Global features extracted and cached")

In [ ]:
# Cell 2.4: Query Expansion (Optional)

def query_expansion(features, k=5, alpha=0.5):
    """Query expansion via k-nearest neighbors averaging."""
    sim_matrix = np.dot(features, features.T)
    knn_indices = np.argsort(sim_matrix, axis=1)[:, -k:]
    
    expanded = np.zeros_like(features)
    for i in range(features.shape[0]):
        expanded[i] = features[i] + alpha * np.mean(features[knn_indices[i]], axis=0)
    
    return normalize(expanded, axis=1, norm="l2")


# Apply query expansion
global_features_expanded = {}
for species, features in global_features_cache.items():
    cfg = SPECIES_CONFIG[species]
    expanded = query_expansion(features, k=cfg["qe_k"])
    global_features_expanded[species] = expanded
    print(f"{species}: QE with k={cfg['qe_k']}")

print("✓ Query expansion applied to global features")

## Section 3: Local Features (SIFT, SuperPoint, ALIKED, DISK)

In [ ]:
# Cell 3.1: SIFT Extractor

class SIFTExtractor:
    """SIFT feature extractor using OpenCV (rotation-invariant)."""
    def __init__(self, max_keypoints=1000):
        self.max_keypoints = max_keypoints
        self.sift = cv2.SIFT_create(nfeatures=max_keypoints)
    
    def extract(self, image_path):
        """Extract SIFT keypoints and descriptors."""
        img = cv2.imread(str(image_path))
        if img is None:
            return None
        
        # Convert to grayscale for SIFT
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # Detect and compute
        keypoints, descriptors = self.sift.detectAndCompute(gray, None)
        
        if descriptors is None or len(keypoints) < 4:
            return None
        
        # RootSIFT: L1-normalize then element-wise sqrt (HotSpotter's descriptor).
        # More discriminative for texture-heavy patterns (turtles, lizards).
        # SIFT descriptors are non-negative (gradient-magnitude histograms).
        descriptors /= (descriptors.sum(axis=1, keepdims=True) + 1e-7)
        descriptors = np.sqrt(descriptors)
        
        # Filter keypoints to animal region using segmentation mask.
        # SIFT runs on the ORIGINAL image (no white-bg boundary artifacts).
        # get_seg_mask() returns None for uncached images → all kpts kept.
        seg_mask = get_seg_mask(image_path)
        if seg_mask is not None:
            h, w = seg_mask.shape
            keep = [
                i for i, kp in enumerate(keypoints)
                if 0 <= int(kp.pt[1]) < h
                and 0 <= int(kp.pt[0]) < w
                and seg_mask[int(kp.pt[1]), int(kp.pt[0])] == 1
            ]
            if len(keep) >= 4:
                keypoints   = [keypoints[i] for i in keep]
                descriptors = descriptors[keep]
        
        # Convert keypoints to numpy array (x, y)
        kpts_array = np.array([kp.pt for kp in keypoints], dtype=np.float32)
        scores_array = np.array([kp.response for kp in keypoints], dtype=np.float32)
        
        return {
            'keypoints': kpts_array,
            'descriptors': descriptors.astype(np.float32),
            'scores': scores_array
        }

print("✓ SIFT extractor defined")

In [ ]:
# Cell 3.2: Feature Extractor Strategy

# SIMPLIFIED APPROACH: SIFT-Only Ensemble
# 
# Due to kornia compatibility issues (ALIKED/DISK not available in all versions),
# we use a proven, robust approach:
#
# 1. SIFT (OpenCV): Classical rotation-invariant features
#    - Works on all platforms
#    - 128-dim descriptors
#    - Matched with BFMatcher + Lowe's ratio test
#    - Excellent for rotation/scale invariance
#
# 2. Global (MiewID v3): Deep learning embeddings
#    - 2152-dim features
#    - Pre-trained on wildlife data
#    - Excellent for overall similarity
#
# Ensemble: Global (65-70%) + SIFT (30-35%)
# This provides a good balance of deep learning and classical CV approaches

print("✓ Using SIFT + MiewID v3 ensemble (robust, compatible approach)")

In [ ]:
# Cell 3.3: ALIKED Extractor (Kornia)

class ALIKEDExtractor:
    """ALIKED feature extractor via kornia (deformation-robust)."""
    def __init__(self, max_keypoints=1024, device='cuda'):
        self.device = device
        self.extractor = KF.ALIKED(max_num_keypoints=max_keypoints).to(device).eval()
    
    def extract(self, image_path):
        """Extract ALIKED keypoints and descriptors."""
        img = cv2.imread(str(image_path))
        if img is None:
            return None
        
        # Convert to grayscale tensor
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        tensor = torch.from_numpy(gray).float()[None, None] / 255.0
        tensor = tensor.to(self.device)
        
        with torch.no_grad():
            result = self.extractor(tensor)
        
        if result is None or len(result[0]['keypoints']) == 0:
            return None
        
        return {
            'keypoints': result[0]['keypoints'].cpu().numpy(),
            'descriptors': result[0]['descriptors'].cpu().numpy(),
            'scores': result[0]['scores'].cpu().numpy()
        }

print("✓ ALIKED extractor defined")

In [ ]:
# Cell 3.4: KAZE Extractor (OpenCV)
#
# KAZE uses a non-linear (anisotropic diffusion) scale space instead of the
# Gaussian pyramid used by SIFT. This gives keypoints that are more stable
# near object boundaries and textured regions.
#
# Advantages over ALIKED/DISK for this environment:
#   - Pure OpenCV: no kornia, no pretrained weights, no internet needed
#   - Float descriptors (64-dim): compatible with existing BFMatcher + L2
#   - SAM3 mask filter applied (SeaTurtle has 100% coverage)
#   - RootSIFT does NOT apply: KAZE uses M-SURF-style descriptors with signed
#     Haar wavelet responses (sum_dx, sum_dy can be negative). sqrt(negative) = NaN.
#
# Note: KAZE has no built-in max_features parameter. We sort by keypoint
# response and keep the top max_keypoints after detection.

class KAZEExtractor:
    """KAZE feature extractor using OpenCV (non-linear scale space, float descriptors)."""
    def __init__(self, max_keypoints=1000):
        self.max_keypoints = max_keypoints
        self.kaze = cv2.KAZE_create(upright=False)  # upright=False = rotation invariant

    def extract(self, image_path):
        """Extract KAZE keypoints and descriptors with SAM3 mask filter (no RootSIFT)."""
        img = cv2.imread(str(image_path))
        if img is None:
            return None

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        keypoints, descriptors = self.kaze.detectAndCompute(gray, None)

        if descriptors is None or len(keypoints) < 4:
            return None

        # Limit to max_keypoints by response strength (KAZE has no nfeatures param)
        if len(keypoints) > self.max_keypoints:
            sorted_pairs = sorted(enumerate(keypoints), key=lambda x: -x[1].response)
            keep_idx = sorted([i for i, _ in sorted_pairs[:self.max_keypoints]])
            keypoints   = [keypoints[i] for i in keep_idx]
            descriptors = descriptors[keep_idx]

        # NOTE: RootSIFT (L1-norm + sqrt) is NOT applied to KAZE descriptors.
        # KAZE uses an M-SURF-style descriptor that stores signed Haar wavelet
        # responses (sum_dx, sum_dy can be negative). Applying sqrt to negative
        # values produces NaN → silent zero-match failure.
        # KAZE descriptors are L2-normalised internally by OpenCV; use as-is.
        descriptors = descriptors.astype(np.float32)

        # SAM3 mask filter: discard keypoints outside the animal region.
        # SeaTurtleID2022 has 100% cache coverage → every image benefits.
        seg_mask = get_seg_mask(image_path)
        if seg_mask is not None:
            h, w = seg_mask.shape
            keep = [
                i for i, kp in enumerate(keypoints)
                if 0 <= int(kp.pt[1]) < h
                and 0 <= int(kp.pt[0]) < w
                and seg_mask[int(kp.pt[1]), int(kp.pt[0])] == 1
            ]
            if len(keep) >= 4:
                keypoints   = [keypoints[i] for i in keep]
                descriptors = descriptors[keep]

        kpts_array   = np.array([kp.pt       for kp in keypoints], dtype=np.float32)
        scores_array = np.array([kp.response for kp in keypoints], dtype=np.float32)

        return {
            'keypoints':   kpts_array,
            'descriptors': descriptors.astype(np.float32),
            'scores':      scores_array,
        }

print("✓ KAZE extractor defined (OpenCV, no pretrained weights, no RootSIFT)")


In [ ]:
# Cell 3.5: Extract Local Features per Species

def get_extractor(extractor_name, device='cuda'):
    """Factory function to create feature extractors."""
    if extractor_name == 'sift':
        return SIFTExtractor(max_keypoints=1000)
    elif extractor_name == 'kaze':
        return KAZEExtractor(max_keypoints=1000)
    else:
        raise ValueError(f"Unknown extractor: {extractor_name}. Supported: 'sift', 'kaze'.")


def extract_local_features_for_dataset(species, extractor_name, dataset_meta, root_dir):
    """Extract local features for all images in a dataset split."""
    cache_file = f"cache/local_features/{species}_{extractor_name}.pkl"
    
    # Check cache
    if os.path.exists(cache_file):
        print(f"  Loading cached {extractor_name} features: {cache_file}")
        with open(cache_file, 'rb') as f:
            return pickle.load(f)
    
    # Extract features
    print(f"  Extracting {extractor_name} features for {len(dataset_meta)} images...")
    extractor = get_extractor(extractor_name, DEVICE)
    
    features_list = []
    for idx, row in tqdm(dataset_meta.iterrows(), total=len(dataset_meta), leave=False, desc=f"{extractor_name}"):
        img_path = os.path.join(root_dir, row['path'])
        feats = extractor.extract(img_path)
        features_list.append(feats)
    
    # Cache to disk
    with open(cache_file, 'wb') as f:
        pickle.dump(features_list, f)
    print(f"  Cached to {cache_file}")
    
    return features_list


# Extract local features for each species
local_features_cache = {}

for species in test_meta["dataset"].unique():
    print(f"\n{'='*60}")
    print(f"Processing {species} - Local Features (SIFT)")
    
    cfg = SPECIES_CONFIG[species]
    sp_meta = test_meta[test_meta["dataset"] == species]
    
    local_features_cache[species] = {}
    for extractor_name in cfg["local_extractors"]:
        features = extract_local_features_for_dataset(
            species, extractor_name, sp_meta, ROOT_DIR
        )
        local_features_cache[species][extractor_name] = features
        
        # Stats
        valid_count = sum(1 for f in features if f is not None)
        if valid_count > 0:
            avg_kpts = np.mean([len(f['keypoints']) for f in features if f is not None])
            print(f"    ✓ {extractor_name}: {valid_count}/{len(features)} valid, "
                  f"avg {avg_kpts:.0f} keypoints")
        else:
            print(f"    ⚠ {extractor_name}: No valid features extracted!")

print("\n✓ Local features extracted and cached")

In [ ]:
# Cell 3.6: Verify Local Features Cache

print("Local Features Summary:")
for species, extractors_dict in local_features_cache.items():
    print(f"\n{species}:")
    for extractor_name, features_list in extractors_dict.items():
        valid = sum(1 for f in features_list if f is not None)
        print(f"  {extractor_name}: {valid}/{len(features_list)} images with features")

print("\n✓ Local features verified")

## Section 4: Matching with LightGlue

In [ ]:
# Cell 4.1: SIFT Matcher (BFMatcher)

class SIFTMatcher:
    """SIFT matcher using BFMatcher with Lowe's ratio test."""
    def __init__(self, device='cuda'):
        # BFMatcher for SIFT (L2 norm)
        self.matcher = cv2.BFMatcher(cv2.NORM_L2)
    
    def match(self, feat0, feat1):
        """Match SIFT keypoints between two images."""
        if feat0 is None or feat1 is None:
            return {'matches': np.array([]), 'match_indices': np.array([]), 'confidence': np.array([])}
        
        if len(feat0['descriptors']) < 2 or len(feat1['descriptors']) < 2:
            return {'matches': np.array([]), 'match_indices': np.array([]), 'confidence': np.array([])}
        
        # kNN matching with k=2
        matches = self.matcher.knnMatch(feat0['descriptors'], feat1['descriptors'], k=2)
        
        # Lowe's ratio test (0.75 threshold)
        good_matches = []
        for pair in matches:
            if len(pair) == 2:
                m, n = pair
                if m.distance < 0.75 * n.distance:
                    good_matches.append(m)
        
        if len(good_matches) == 0:
            return {
                'matches': np.array([], dtype=int),
                'match_indices': np.array([], dtype=int),
                'confidence': np.array([], dtype=np.float32)
            }
        
        return {
            'matches': np.array([m.queryIdx for m in good_matches], dtype=int),
            'match_indices': np.array([m.trainIdx for m in good_matches], dtype=int),
            'confidence': 1.0 - np.array([m.distance for m in good_matches], dtype=np.float32) / 255.0
        }

print("✓ SIFT matcher defined (BFMatcher with Lowe's ratio test)")

In [ ]:
# Cell 4.2: Ultra-Fast GPU Batch Matching

def batch_sift_match_gpu(desc1, desc2, ratio_thresh=0.75):
    """GPU-accelerated SIFT matching using PyTorch batch operations."""
    if desc1 is None or desc2 is None or len(desc1) < 2 or len(desc2) < 2:
        return 0
    
    # Convert to torch tensors on GPU
    d1 = torch.from_numpy(desc1).float().to(DEVICE)
    d2 = torch.from_numpy(desc2).float().to(DEVICE)
    
    # Compute pairwise L2 distances (batch operation on GPU)
    # distances[i, j] = L2 distance between d1[i] and d2[j]
    distances = torch.cdist(d1, d2, p=2)  # Shape: (N1, N2)
    
    # Get 2 nearest neighbors for each descriptor in d1
    topk_dists, topk_indices = torch.topk(distances, k=min(2, d2.shape[0]), 
                                          dim=1, largest=False)
    
    # Lowe's ratio test (vectorized)
    if topk_dists.shape[1] == 2:
        ratios = topk_dists[:, 0] / (topk_dists[:, 1] + 1e-8)
        good_matches = (ratios < ratio_thresh).sum().item()
    else:
        good_matches = topk_dists.shape[0]  # All are good if only 1 neighbor
    
    return good_matches


def compute_pairwise_matches_fast(features_list, extractor_type, species):
    """Ultra-fast pairwise matching using GPU batch operations."""
    n = len(features_list)
    cache_file = f"cache/match_scores/{species}_{extractor_type}_matches.npy"
    
    # Check cache FIRST
    if os.path.exists(cache_file):
        print(f"  ✓ Loading cached {extractor_type} match scores: {cache_file}")
        return np.load(cache_file)
    
    print(f"  Computing {extractor_type} matches for {n} images (GPU batch mode)...")
    
    # Top-K strategy: only match to most similar candidates
    K = min(100, n)
    
    # Get global similarity for candidate selection
    global_feats = global_features_expanded[species]
    global_sim = np.dot(global_feats, global_feats.T)
    
    # Pre-compute top-K candidates for ALL images at once
    top_k_all = np.argsort(global_sim, axis=1)[:, -K:]  # Shape: (N, K)
    
    # Initialize match matrix
    match_matrix = np.zeros((n, n), dtype=np.float32)
    np.fill_diagonal(match_matrix, 1.0)
    
    # Process in batches for GPU efficiency
    batch_size = 50  # Process 50 query images at once
    
    for start_idx in tqdm(range(0, n, batch_size), desc=f"Matching {extractor_type}"):
        end_idx = min(start_idx + batch_size, n)
        
        # For each query in this batch
        for i in range(start_idx, end_idx):
            if features_list[i] is None:
                continue
            
            # Get top-K candidates for this query
            candidates = top_k_all[i]
            
            # Batch match against all candidates (GPU accelerated)
            for j in candidates:
                if i == j or match_matrix[i, j] > 0:
                    continue
                
                if features_list[j] is None:
                    continue
                
                # GPU batch matching
                num_matches = batch_sift_match_gpu(
                    features_list[i]['descriptors'],
                    features_list[j]['descriptors']
                )
                
                # Simple scoring: sigmoid of match count
                score = 1.0 - np.exp(-num_matches / 20.0)
                
                match_matrix[i, j] = score
                match_matrix[j, i] = score
    
    # Cache to disk
    np.save(cache_file, match_matrix)
    print(f"  ✓ Cached to {cache_file}")
    
    return match_matrix

print("✓ Ultra-fast GPU batch matching defined")

In [ ]:
# Cell 4.3: RANSAC Verification and Match Scoring

def ransac_filter(kpts0, kpts1, matches, threshold=3.0):
    """RANSAC geometric verification of matches."""
    if len(matches['matches']) < 4:
        return np.array([], dtype=int)
    
    pts0 = kpts0[matches['matches']]
    pts1 = kpts1[matches['match_indices']]
    
    # Estimate fundamental matrix with RANSAC
    _, mask = cv2.findFundamentalMat(
        pts0, pts1, cv2.FM_RANSAC, threshold, confidence=0.99
    )
    
    if mask is None:
        return np.array([], dtype=int)
    
    return np.where(mask.ravel())[0]


def score_local_match(matches, feat0, feat1):
    """Convert matches to similarity score [0, 1]."""
    if feat0 is None or feat1 is None:
        return 0.0
    
    if len(matches['matches']) == 0:
        return 0.0
    
    # RANSAC geometric verification
    inlier_indices = ransac_filter(
        feat0['keypoints'], 
        feat1['keypoints'], 
        matches
    )
    
    num_inliers = len(inlier_indices)
    if num_inliers == 0:
        return 0.0
    
    # Average confidence of inlier matches
    avg_confidence = np.mean(matches['confidence'][inlier_indices])
    
    # Sigmoid-like normalization (plateaus at ~50 matches)
    match_score = 1.0 - np.exp(-num_inliers / 20.0)
    
    # Combine match count and confidence
    final_score = match_score * avg_confidence
    
    return float(final_score)

print("✓ RANSAC verification and match scoring functions defined")

In [ ]:
# Cell 4.4: Compute and Cache Match Scores

match_scores_cache = {}

for species in test_meta["dataset"].unique():
    print(f"\n{'='*60}")
    print(f"Processing {species} - Match Scores")
    
    cfg = SPECIES_CONFIG[species]
    match_scores_cache[species] = {}
    
    for extractor_name in cfg["local_extractors"]:
        features_list = local_features_cache[species][extractor_name]
        
        # Use fast top-K matching strategy
        match_matrix = compute_pairwise_matches_fast(features_list, extractor_name, species)
        match_scores_cache[species][extractor_name] = match_matrix
        
        # Stats
        non_diag = match_matrix[~np.eye(match_matrix.shape[0], dtype=bool)]
        print(f"    ✓ {extractor_name}: Shape {match_matrix.shape}, "
              f"Mean score: {non_diag.mean():.3f}, "
              f"Non-zero pairs: {(non_diag > 0).sum():,}")

print("\n✓ Match scores computed and cached")

## Section 5: Ensemble Voting

In [ ]:
# Cell 5.1: Ensemble Scoring Function

def compute_ensemble_similarity_matrix(species):
    """Compute weighted ensemble similarity matrix."""
    cfg = SPECIES_CONFIG[species]
    
    # Global similarity (cosine)
    global_feats = global_features_expanded[species]
    global_sim = np.dot(global_feats, global_feats.T)
    
    # Start with weighted global similarity
    ensemble_sim = cfg["global_weight"] * global_sim
    
    # Add weighted local match scores
    for extractor_name, weight in cfg["local_weights"].items():
        local_sim = match_scores_cache[species][extractor_name]
        ensemble_sim += weight * local_sim
    
    return ensemble_sim

print("✓ Ensemble scoring function defined")

In [ ]:
# Cell 5.2: Compute Ensemble Scores for All Species

ensemble_similarity_cache = {}

for species in test_meta["dataset"].unique():
    print(f"\nComputing ensemble scores for {species}...")
    ensemble_sim = compute_ensemble_similarity_matrix(species)
    ensemble_similarity_cache[species] = ensemble_sim
    
    # Stats
    print(f"  Shape: {ensemble_sim.shape}")
    print(f"  Mean similarity: {ensemble_sim.mean():.3f}")
    print(f"  Std similarity: {ensemble_sim.std():.3f}")
    print(f"  Min/Max: {ensemble_sim.min():.3f} / {ensemble_sim.max():.3f}")

print("\n✓ Ensemble similarity matrices computed")

In [ ]:
# Cell 5.3: Weighted Voting Summary

print("Ensemble Weights Summary:\n")
for species, cfg in SPECIES_CONFIG.items():
    print(f"{species}:")
    print(f"  Global (MiewID v3): {cfg['global_weight']:.0%}")
    for extractor, weight in cfg['local_weights'].items():
        print(f"  {extractor.upper()}: {weight:.0%}")
    print()

print("✓ Ensemble voting configured")

In [ ]:
# Cell 5.4: Weight + Threshold Calibration using Training Identity Labels
# ───────────────────────────────────────────────────────────────────────
# For species with training splits (Lynx, Salamander, SeaTurtle) we jointly
# calibrate global_weight, sift_weight, kaze_weight and threshold_cluster
# by grid-searching AMI against known training identities.
#
# Pipeline:
#   1. Subsample ≤500 training images per species.
#   2. Extract MiewID global embeddings (GPU, TTA).
#   3. Extract SIFT + KAZE descriptors (CPU, RootSIFT for SIFT).
#   4. Compute BFMatcher match matrices using top-50 global preselection.
#   5. Grid-search (gw, sw, thr) with kw = 1-gw-sw to maximise AMI.
#   6. Write optimal weights + threshold back into SPECIES_CONFIG.
#
# THL: no training split → weights + threshold unchanged from V2.5.
# Calibrated values are used by Cell 6.2 (AgglomerativeClustering).

import time as _time
from PIL import Image as _PILImg
from sklearn.metrics import adjusted_mutual_info_score as _ami_score

CALIB_SPECIES   = ["LynxID2025", "SalamanderID2025", "SeaTurtleID2022"]
CALIB_MAX_IMGS  = 500    # subsample if species has more training images
CALIB_BFM_TOPK  = 50    # top-K preselection per image for BFMatcher
CALIB_KPT_CAP   = 300   # max keypoints/image during calibration (speed)

GLOBAL_W_GRID   = [0.55, 0.60, 0.65, 0.70, 0.75]
SIFT_W_GRID     = [0.10, 0.15, 0.20, 0.25]
THR_GRID        = [round(0.15 + i * 0.02, 2) for i in range(23)]  # 0.15..0.59

print("=" * 65)
print("V2.8 -- Joint Weight + Threshold Calibration (Training Identities)")
print("=" * 65)
print(f"  Grid: gw={GLOBAL_W_GRID}  sw={SIFT_W_GRID}")
print(f"  Thresholds: {THR_GRID[0]:.2f} .. {THR_GRID[-1]:.2f} ({len(THR_GRID)} steps)")
print(f"  Max training images / species: {CALIB_MAX_IMGS}")


# ── Helper 1: global embedding extraction ────────────────────────────────────
def _extract_calib_embs(model, image_paths, image_size, batch_size=64):
    '''Extract L2-normalised MiewID embeddings (original + hflip TTA).'''
    _tfm = T.Compose([
        T.Resize((image_size, image_size)),
        T.ToTensor(),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])
    all_feats = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(image_paths), batch_size):
            batch_paths = image_paths[i : i + batch_size]
            imgs = []
            for p in batch_paths:
                try:
                    img = _PILImg.open(p).convert("RGB")
                    imgs.append(_tfm(img))
                except Exception:
                    imgs.append(torch.zeros(3, image_size, image_size))
            batch = torch.stack(imgs).to(DEVICE)
            with torch.cuda.amp.autocast():
                feats = model(batch) + model(torch.flip(batch, dims=[3]))
            feats = torch.nn.functional.normalize(feats.float(), p=2, dim=1)
            all_feats.append(feats.cpu().numpy())
    return np.concatenate(all_feats).astype(np.float32)


# ── Helper 2: local descriptor extraction (SIFT or KAZE) ─────────────────────
def _extract_calib_local(img_paths, img_keys, ext_type):
    '''Extract SIFT or KAZE descriptors for a list of training image paths.

    img_keys: relative paths (e.g. "LynxID2025/img001.jpg") used for
              get_seg_mask() lookup.  Returns None if not in SAM3 cache.
    RootSIFT is applied to SIFT descriptors (L1-norm + sqrt).
    Returns: list of np.ndarray (N_kpts, D) float32  or  None.
    '''
    if ext_type == "sift":
        det = cv2.SIFT_create(nfeatures=CALIB_KPT_CAP)
    else:
        det = cv2.KAZE_create(upright=False, threshold=0.001,
                              nOctaves=4, nOctaveLayers=4)

    all_descs = []
    for path, key in zip(img_paths, img_keys):
        try:
            img_rgb  = np.array(_PILImg.open(path).convert("RGB"))
            gray     = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
            seg_mask = get_seg_mask(path)  # absolute path → SAM3 + V2.8 yellow filter
            kps, descs = det.detectAndCompute(gray, None)
            if descs is None or len(descs) < 4:
                all_descs.append(None)
                continue
            # Filter keypoints to animal region where mask is available
            if seg_mask is not None:
                h, w = seg_mask.shape[:2]
                pts   = np.array([[int(kp.pt[0]), int(kp.pt[1])] for kp in kps])
                xs    = np.clip(pts[:, 0], 0, w - 1)
                ys    = np.clip(pts[:, 1], 0, h - 1)
                valid = seg_mask[ys, xs] > 0
                descs = descs[valid]
                if len(descs) < 4:
                    all_descs.append(None)
                    continue
            # RootSIFT for SIFT descriptors (KAZE has signed float descs)
            if ext_type == "sift":
                descs = descs.astype(np.float32)
                descs /= (descs.sum(axis=1, keepdims=True) + 1e-7)
                descs  = np.sqrt(descs)
            if len(descs) > CALIB_KPT_CAP:
                descs = descs[:CALIB_KPT_CAP]
            all_descs.append(descs.astype(np.float32))
        except Exception:
            all_descs.append(None)
    return all_descs


# ── Helper 3: BFMatcher match matrix with global top-K preselection ──────────
def _calib_match_matrix(descs_list, global_embs, top_k):
    '''Build (N, N) float32 match matrix via BFMatcher + Lowe ratio test.

    Only evaluates pairs in each image's top-K by global cosine similarity
    (same strategy as the main pipeline).
    Score = 1 - exp(-good_matches / 20).
    '''
    N      = len(descs_list)
    matrix = np.zeros((N, N), dtype=np.float32)
    np.fill_diagonal(matrix, 1.0)

    # Collect unique (i<j) pairs to evaluate
    g_sim = np.clip(global_embs @ global_embs.T, 0.0, 1.0)
    pairs = set()
    for i in range(N):
        sims_i      = g_sim[i].copy()
        sims_i[i]   = -1.0
        top_j       = np.argsort(sims_i)[-top_k:]
        for j in top_j:
            pairs.add((min(i, j), max(i, j)))

    bfm = cv2.BFMatcher(cv2.NORM_L2)
    for i, j in pairs:
        di, dj = descs_list[i], descs_list[j]
        if di is None or dj is None or len(di) < 2 or len(dj) < 2:
            continue
        try:
            matches = bfm.knnMatch(di, dj, k=2)
            good    = sum(
                1 for m_pair in matches
                if len(m_pair) == 2 and m_pair[0].distance < 0.75 * m_pair[1].distance
            )
            score           = 1.0 - np.exp(-good / 20.0)
            matrix[i, j]    = score
            matrix[j, i]    = score
        except Exception:
            pass
    return matrix


# ── Main calibration loop ─────────────────────────────────────────────────────
calib_model = get_global_model()
calib_model.eval()

calibrated_config = {}

for sp in CALIB_SPECIES:
    t0 = _time.time()

    sp_train = train_meta[train_meta["dataset"] == sp].copy()
    total_n  = len(sp_train)
    if len(sp_train) > CALIB_MAX_IMGS:
        sp_train = sp_train.sample(n=CALIB_MAX_IMGS, random_state=42).reset_index(drop=True)

    true_labels, _ = pd.factorize(sp_train["identity"].values)
    n_ids          = len(np.unique(true_labels))
    img_paths      = [os.path.join(ROOT_DIR, p) for p in sp_train["path"].values]
    img_keys       = list(sp_train["path"].values)

    print(f"\n{sp}: {len(sp_train)}/{total_n} train images, {n_ids} identities")

    cfg = SPECIES_CONFIG[sp]

    # Step 1: global embeddings
    print(f"  [1/4] Extracting global embeddings...")
    global_embs = _extract_calib_embs(calib_model, img_paths, cfg["image_size"])
    global_sim  = np.clip(global_embs @ global_embs.T, 0.0, 1.0).astype(np.float32)

    # Step 2: SIFT match matrix
    print(f"  [2/4] Extracting SIFT descriptors + BFMatcher...")
    sift_descs  = _extract_calib_local(img_paths, img_keys, "sift")
    sift_matrix = _calib_match_matrix(sift_descs, global_embs, CALIB_BFM_TOPK)

    # Step 3: KAZE match matrix
    print(f"  [3/4] Extracting KAZE descriptors + BFMatcher...")
    kaze_descs  = _extract_calib_local(img_paths, img_keys, "kaze")
    kaze_matrix = _calib_match_matrix(kaze_descs, global_embs, CALIB_BFM_TOPK)

    # Step 4: grid search
    n_combos = sum(
        1 for gw in GLOBAL_W_GRID for sw in SIFT_W_GRID
        if round(1.0 - gw - sw, 4) >= 0.05
    )
    print(f"  [4/4] Grid search ({n_combos} weight combos x {len(THR_GRID)} thresholds)...")

    best_gw  = cfg["global_weight"]
    best_sw  = cfg["local_weights"]["sift"]
    best_kw  = cfg["local_weights"].get("kaze", 0.0)
    best_thr = cfg["threshold_cluster"]
    best_ami = -1.0

    for gw in GLOBAL_W_GRID:
        for sw in SIFT_W_GRID:
            kw = round(1.0 - gw - sw, 4)
            if kw < 0.05:
                continue
            ensemble = gw * global_sim + sw * sift_matrix + kw * kaze_matrix
            for thr in THR_GRID:
                dist = np.clip(1.0 - ensemble, 0.0, 1.0).astype(np.float64)
                pred = AgglomerativeClustering(
                    n_clusters=None, metric="precomputed",
                    linkage="average", distance_threshold=thr,
                ).fit_predict(dist)
                ami = _ami_score(true_labels, pred)
                if ami > best_ami:
                    best_ami, best_gw, best_sw, best_kw, best_thr = ami, gw, sw, kw, thr

    calibrated_config[sp] = {
        "global_weight": best_gw,
        "sift_weight":   best_sw,
        "kaze_weight":   best_kw,
        "threshold":     best_thr,
        "ami":           best_ami,
    }

    prev_gw  = cfg["global_weight"]
    prev_sw  = cfg["local_weights"]["sift"]
    prev_kw  = cfg["local_weights"].get("kaze", 0.0)
    prev_thr = cfg["threshold_cluster"]
    print(f"  weights:   ({prev_gw:.2f}, {prev_sw:.2f}, {prev_kw:.2f})"
          f" -> ({best_gw:.2f}, {best_sw:.2f}, {best_kw:.2f})")
    print(f"  threshold: {prev_thr:.2f} -> {best_thr:.2f}"
          f"  (AMI={best_ami:.4f}, t={_time.time()-t0:.1f}s)")

del calib_model
torch.cuda.empty_cache()

# THL: no training split -- keep V2.5 values unchanged
thl_cfg = SPECIES_CONFIG["TexasHornedLizards"]
calibrated_config["TexasHornedLizards"] = {
    "global_weight": thl_cfg["global_weight"],
    "sift_weight":   thl_cfg["local_weights"]["sift"],
    "kaze_weight":   thl_cfg["local_weights"]["kaze"],
    "threshold":     thl_cfg["threshold_cluster"],
    "ami":           None,
}
print(f"\nTexasHornedLizards: no train split -> weights + threshold unchanged")

# Write calibrated weights + thresholds back into SPECIES_CONFIG (used by Cell 6.2)
for sp, cal in calibrated_config.items():
    SPECIES_CONFIG[sp]["global_weight"]         = cal["global_weight"]
    SPECIES_CONFIG[sp]["local_weights"]["sift"] = cal["sift_weight"]
    if "kaze" in SPECIES_CONFIG[sp]["local_weights"]:
        SPECIES_CONFIG[sp]["local_weights"]["kaze"] = cal["kaze_weight"]
    SPECIES_CONFIG[sp]["threshold_cluster"]     = cal["threshold"]

print("\nCalibrated SPECIES_CONFIG:")
for sp, cfg in SPECIES_CONFIG.items():
    lw = cfg["local_weights"]
    print(f"  {sp:<25}: gw={cfg['global_weight']:.2f}"
          f"  sw={lw.get('sift', 0):.2f}"
          f"  kw={lw.get('kaze', 0):.2f}"
          f"  thr={cfg['threshold_cluster']:.2f}")
print("\n✓ Weights + thresholds updated -- Cell 6.2 will use these values")


## Section 6: Clustering and Submission

In [ ]:
# Cell 6.1: 2-Phase Clustering (Known + Unknown)

def cluster_with_ensemble_scores(species, similarity_matrix, image_ids, threshold):
    """Cluster images using ensemble similarity scores."""
    print(f"\n  Clustering {species}:")
    print(f"    Threshold: {threshold}")
    print(f"    Images: {len(image_ids)}")
    
    # Convert similarity to distance
    dist_matrix = np.clip(1.0 - similarity_matrix, 0, 1)
    
    # Agglomerative clustering
    clustering = AgglomerativeClustering(
        n_clusters=None,
        metric="precomputed",
        linkage="average",
        distance_threshold=threshold,
    )
    labels = clustering.fit_predict(dist_matrix)
    
    n_clusters = len(np.unique(labels))
    print(f"    ✅ Found {n_clusters} individuals")
    
    # Format cluster labels
    cluster_labels = [f"cluster_{species}_{lbl}" for lbl in labels]
    
    return pd.DataFrame({
        "image_id": image_ids,
        "cluster": cluster_labels
    })

print("✓ Clustering function defined")

In [ ]:
# Cell 6.2: Generate Clusters for All Species

results = []

print("="*60)
print("Clustering with Ensemble Scores")
print("="*60)

for species in test_meta["dataset"].unique():
    cfg = SPECIES_CONFIG[species]
    sp_meta = test_meta[test_meta["dataset"] == species]
    
    # Get ensemble similarity matrix
    similarity_matrix = ensemble_similarity_cache[species]
    
    # Cluster
    cluster_df = cluster_with_ensemble_scores(
        species,
        similarity_matrix,
        sp_meta["image_id"].values,
        cfg["threshold_cluster"]
    )
    
    results.append(cluster_df)

print("\n✓ All species clustered")

In [ ]:
# Cell 6.3: Generate Submission CSV

# Combine all results
predictions = pd.concat(results, ignore_index=True)

# Load sample submission
sample_sub = pd.read_csv(os.path.join(ROOT_DIR, "sample_submission.csv"))

# Map predictions to sample submission format
pred_map = dict(zip(predictions["image_id"], predictions["cluster"]))
sample_sub["cluster"] = sample_sub["image_id"].map(pred_map).fillna("cluster_error_0")

# Save submission
sample_sub.to_csv("submission.csv", index=False)

print("\n" + "="*60)
print("🏆 Submission Generated!")
print("="*60)
print(f"Total predictions: {len(sample_sub):,}")
print(f"Total clusters: {sample_sub['cluster'].nunique():,}")
print(f"\nBreakdown by species:")
for species in test_meta["dataset"].unique():
    sp_clusters = predictions[predictions["cluster"].str.contains(species)]["cluster"].nunique()
    print(f"  {species}: {sp_clusters} clusters")

print("\n✓ submission.csv saved")
sample_sub.head(10)